In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

: 

# 01 · Caso de negocio — LibranzaRadar

**LibranzaRadar** apoya a una caja de compensación a modernizar su operación de libranza. Hoy, al aprobar un crédito,
el analista no sabe cuáles tienen más probabilidad de caer en mora.

| | |
|---|---|
| **Quién decide** | El analista de crédito (conserva la decisión final) |
| **Qué decide con el modelo** | A qué créditos aprobados dar **revisión y seguimiento prioritario** con la empresa pagadora, en el portal de autogestión |
| **Con qué información** | Solo la disponible **el día de la aprobación** (condiciones del crédito, producto, empresa pagadora, ciudad, garantía, codeudor) |
| **Objetivo** | `mora_30` = 1 si el crédito supera 30 días de mora |
| **Métrica principal** | Recall de la clase "en mora" con **precisión ≥ 20 %** |
| **Error más costoso** | Falso negativo: no detectar un crédito que caerá en mora (pérdida de cartera y costos de cobranza) |

La ficha completa está en `reports/00_caso.md`. Este notebook comprueba con los datos que la ficha es viable.

In [2]:
raw = pd.read_csv(ROOT / "data" / "raw" / "Base_sintetica_creditos.csv", low_memory=False)
print(f"filas: {len(raw):,} · columnas: {raw.shape[1]}")
print(f"clientes distintos: {raw['NUMERO DE IDENTIFICACION'].nunique():,}")
varios = (raw.groupby('NUMERO DE IDENTIFICACION').size() > 1).sum()
print(f"clientes con más de un crédito: {varios:,}  -> la partición debe ser por CLIENTE")
y = (raw["DIAS DE MORA"] > 30).astype(int)
print(f"créditos con más de 30 días de mora: {y.mean():.1%}")
vacias = [c for c in raw.columns if raw[c].nunique(dropna=True) <= 1]
print(f"columnas vacías o constantes: {len(vacias)} de {raw.shape[1]}")

filas: 60,000 · columnas: 189
clientes distintos: 41,855
clientes con más de un crédito: 12,627  -> la partición debe ser por CLIENTE
créditos con más de 30 días de mora: 19.0%
columnas vacías o constantes: 98 de 189


## Por qué la exactitud no sirve

Con 19 % de créditos en mora, un "modelo" que **nunca** predice mora acierta el 81 % y no detecta ninguno. Por eso la
métrica principal es el **recall de la mora** (cuántas moras detectamos) **exigiendo una precisión mínima** (que no
mandemos a revisión a todo el mundo).

**Precisión mínima = 20 %.** Revisar al azar tiene una precisión igual a la tasa de mora (~16 % después de la limpieza):
exigimos que de cada 5 créditos marcados al menos 1 caiga de verdad en mora, es decir, **25 % mejor que el azar**.
Se fija aquí, **antes** de entrenar cualquier modelo, y no se cambia después.

In [3]:
from proyecto.config import MIN_PRECISION
print(f"Precisión mínima exigida: {MIN_PRECISION:.0%}")
print("Línea base trivial (nunca predecir mora): recall = 0 %, exactitud =", f"{1 - y.mean():.1%}")
print("Línea base 'revisar todo': recall = 100 %, precisión =", f"{y.mean():.1%}", "(< 20 %: no cumple)")

Precisión mínima exigida: 20%
Línea base trivial (nunca predecir mora): recall = 0 %, exactitud = 81.0%
Línea base 'revisar todo': recall = 100 %, precisión = 19.0% (< 20 %: no cumple)
